# CF6 — Info Geometry · Fisher–Ruppeiner Foundations

- Canon (anchor-only; do not duplicate): [../../Complete-Formalisms/CF6_Info_Geom_Fisher_Ruppeiner_Foundations.md](../../Complete-Formalisms/CF6_Info_Geom_Fisher_Ruppeiner_Foundations.md)
- Purpose: step-by-step, runnable walkthrough establishing minimal, falsifiable checks for information-geometry foundations used in the CF6 formalism.

Navigation anchors (canon registries, links for context only):
- GENERIC / metriplectic evolution (background thermodynamic structure): [../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-140](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-140)
- Entropy production (for thermodynamic monotonicity context): [../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-143](../../../z.CANONICAL_Equations/00_EQUATIONS.md#vdm-e-143)

Scope: we do not restate theory here; we show small, testable numerics that support the CF6 formalism’s claims of metric PSD, score–Fisher equality, and coordinate invariance.

## Step 1 — Analytic Fisher information for Normal(μ, σ)

Parameterization θ=(μ, σ) with σ>0. For one i.i.d. sample x from Normal(μ, σ), a standard result is the Fisher information matrix
$$\mathcal I(μ,σ) = \begin{pmatrix} 1/σ^2 & 0 \\ 0 & 2/σ^2 \end{pmatrix}.$$

This must be symmetric, positive semidefinite (actually positive definite for σ>0). We compute it for a given σ and check eigenvalues.

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

def fisher_normal_mu_sigma(sigma: float):
    return np.diag([1.0/sigma**2, 2.0/sigma**2])

sigma = 1.7
I = fisher_normal_mu_sigma(sigma)
eig = np.linalg.eigvalsh(I)
print({'Fisher_matrix': I.tolist(), 'eig_min': float(eig[0]), 'eig_max': float(eig[-1])})
assert eig[0] > 0.0, 'Fisher must be PD for σ>0'


## Step 2 — Score covariance equals Fisher information (Monte Carlo check)

For regular models, $\mathcal I(θ) = \mathrm{Cov}_θ[\nabla_θ \log p_θ(X)]$. We estimate the score covariance empirically and compare to the analytic Fisher matrix above. This provides a falsifiable check: the Frobenius norm of the difference should be small and decay with sample size.

In [ ]:
rng = np.random.default_rng(12345)

mu, sigma = 0.4, 1.2
I_analytic = fisher_normal_mu_sigma(sigma)

def score_normal_mu_sigma(x, mu, sigma):
    # log p = -0.5*log(2πσ^2) - (x-μ)^2 / (2σ^2)
    s_mu = (x - mu) / (sigma**2)
    s_sigma = -1.0/sigma + ((x-mu)**2)/(sigma**3)
    return np.array([s_mu, s_sigma])

N = 200000  # large to make the Monte Carlo variance small but still fast in NumPy
x = rng.normal(loc=mu, scale=sigma, size=N)
S = np.vstack([score_normal_mu_sigma(xi, mu, sigma) for xi in x])  # (N, 2)
S -= S.mean(axis=0, keepdims=True)  # center for covariance
I_emp = (S.T @ S) / (N-1)

diff = I_emp - I_analytic
fro = float(np.linalg.norm(diff, ord='fro'))
print({'I_emp': I_emp.tolist(), 'I_analytic': I_analytic.tolist(), 'frobenius_diff': fro})
assert fro < 0.05, 'Empirical score covariance should be close to analytic Fisher (tolerance heuristic)'


## Step 3 — Coordinate (reparameterization) check via Jacobian transport

Let φ=(μ, s) with s=log σ. Under a smooth reparameterization with Jacobian J=∂θ/∂φ, Fisher transforms as
$$\mathcal I_{\phi} = J^\top \mathcal I_{\theta} J.$$

We test this mapping numerically by computing both sides and comparing.

In [ ]:
mu, sigma = 0.4, 1.2
s = np.log(sigma)

# θ=(μ,σ), φ=(μ,s) with σ = e^s
# Jacobian ∂θ/∂φ = [[∂μ/∂μ, ∂μ/∂s], [∂σ/∂μ, ∂σ/∂s]] = [[1, 0],[0, e^s]]
J = np.array([[1.0, 0.0],[0.0, np.exp(s)]])

I_theta = fisher_normal_mu_sigma(sigma)
I_phi_from_push = J.T @ I_theta @ J

# Compute Fisher in φ directly by changing variables: for Normal(μ, σ) with s=log σ, analytic result:
# I_phi = diag( e^{-2s}, 2 ), since 1/σ^2 = e^{-2s} and 2/σ^2 * (∂σ/∂s)^2 with ∂σ/∂s = σ gives 2/σ^2 * σ^2 = 2.
I_phi_direct = np.diag([np.exp(-2*s), 2.0])

diff_phi = I_phi_from_push - I_phi_direct
fro_phi = float(np.linalg.norm(diff_phi, ord='fro'))
print({'I_phi_from_push': I_phi_from_push.tolist(), 'I_phi_direct': I_phi_direct.tolist(), 'frobenius_diff': fro_phi})
assert fro_phi < 1e-12, 'Reparameterization Fisher transport must match analytic form'


## (Optional) Note on Ruppeiner geometry

Ruppeiner’s metric is (up to sign conventions) the Hessian of entropy with respect to extensive variables; its practical evaluation is model-dependent. This notebook confines itself to the Fisher (statistical) metric checks above. CF6 formalism may specify which thermodynamic potentials/variables are used; their Hessian-based metrics can be checked similarly by finite-difference Hessians and PSD/eigen diagnostics.

## Summary — Minimal, falsifiable info-geometry checks

- Fisher information for Normal(μ,σ) is symmetric PD; eigenvalues are positive.
- Empirical score covariance approaches analytic Fisher; the Frobenius difference is small at large N.
- Reparameterization invariance holds numerically for φ=(μ, log σ) via Jacobian pushforward.

These results are deterministic and sufficient to support the CF6 formalism’s testability without the full proposal pipeline.

## Repro notes

- Determinism: fixed RNG seed for the Monte Carlo step; IEEE-754 double precision assumed.
- No files are written; production artifacts must be routed via `io_paths` per repository policy.

In [ ]:
# io_paths bootstrap (optional, no file writes in this notebook)
from pathlib import Path
import sys
COMMON = Path.cwd().resolve() / 'Derivation' / 'code' / 'common'
if COMMON.exists() and str(COMMON) not in sys.path:
    sys.path.insert(0, str(COMMON))
try:
    from io_paths import figure_path, log_path  # noqa: F401
except Exception as e:
    print('[warn] io_paths not available:', e)
